# End of week 1 exercise

To demonstrate your familiarity with OpenAI API, and also Ollama, build a tool that takes a technical question,  
and responds with an explanation. This is a tool that you will be able to use yourself during the course!

In [19]:
# imports
from openai import OpenAI
from dotenv import load_dotenv
import os
from IPython.display import Markdown, display, update_display

In [10]:
# constants

MODEL_GPT = 'gemini-3.1-flash-lite'
MODEL_LLAMA = 'gemma3:1b'

In [27]:
# set up environment
load_dotenv(override=True)
api_key=os.getenv('GEMINI_API_KEY')
base_url=os.getenv('GEMINI_BASE_URL')

gemini = OpenAI(base_url=base_url, api_key=api_key)
ollama = OpenAI(base_url='http://127.0.0.1:11434/v1', api_key='something')

In [13]:
system_prompt = """
You are a senior software engineer and AI/ML professional with strong expertise in machine learning, deep learning, generative AI, large language models (LLMs), RAG, AI agents, MLOps, and production AI systems. 
You approach problems with the rigor of an experienced engineer and explain technical concepts clearly, accurately, and practically.
"""

In [16]:
# here is the question; type over this to ask something new

question = """
Please explain what this code does and why:
yield from {book.get("author") for book in books if book.get("author")}
"""

In [21]:
# Get gpt-4o-mini to answer, with streaming

messages = [
    { "role": "system", "content" : system_prompt},
    { "role": "user", "content" : question}
]
stream = gemini.chat.completions.create(model=MODEL_GPT, messages=messages, stream=True)
response = ""
display_handle = display(Markdown(content), display_id=True)

for chunk in stream:
    response += chunk.choices[0].delta.content or ''
    update_display(Markdown(response), display_id=display_handle.display_id)


This line of code is a concise and efficient way to **extract a unique, flattened collection of author names** from a list of dictionaries (where each dictionary represents a book).

Here is the breakdown of the syntax and the engineering logic behind it.

### The Breakdown

1.  **`{book.get("author") for book in books if book.get("author")}`**:
    *   **Set Comprehension**: The `{...}` syntax creates a Python `set`. Sets are unordered collections that **automatically handle deduplication**—if two books have the same author, that author will only appear once in the resulting set.
    *   **`book.get("author")`**: Using `.get()` is a defensive programming best practice. It prevents the code from throwing a `KeyError` if a specific book dictionary is missing the "author" key (it returns `None` instead).
    *   **`if book.get("author")`**: This is a filter. It ensures that if the author key is missing (`None`) or the author name is an empty string, that record is excluded from the set.
2.  **`yield from`**:
    *   This is used inside a **generator function**. Instead of returning the entire set at once, `yield from` allows the function to "yield" each element of that set one by one to the caller.
    *   This makes the function a generator, which is memory-efficient because it produces values lazily rather than loading them all into RAM.

---

### Why use this approach?

As a senior engineer, I look at this code and identify three main benefits:

#### 1. Performance (Deduplication)
If `books` contains 10,000 entries, many authors will repeat. By using a `set`, you reduce the total number of items to only the *unique* authors in $O(N)$ time. This is significantly more efficient than checking `if author in list` repeatedly (which would be $O(N^2)$).

#### 2. Memory Efficiency (Lazy Evaluation)
By combining this with `yield from`, you are likely creating a generator. If the consumer of this code only needs to iterate through the authors once (e.g., to display them in a UI or save them to a database), you avoid allocating a massive array in memory. You process items as they are requested.

#### 3. Defensive Programming
The combination of `.get()` and the `if` condition makes the code resilient to "dirty data." In real-world production systems (like those handling JSON from external APIs), you rarely have perfectly formatted data. This snippet safely skips `None` values, empty strings, or missing keys without crashing the program.

### A Practical Example
If your data looks like this:
```python
books = [
    {"title": "The Hobbit", "author": "J.R.R. Tolkien"},
    {"title": "The Silmarillion", "author": "J.R.R. Tolkien"},
    {"title": "Unknown Book"}, 
    {"title": "1984", "author": "George Orwell"}
]
```

**The code would produce:**
`"J.R.R. Tolkien"` then `"George Orwell"` (in an arbitrary order).

### Refinement Note
While this is idiomatic and clever, ensure that the order of authors doesn't matter for your use case. Because a `set` is **unordered**, if you need the authors returned in alphabetical order or by appearance, you would need to wrap the comprehension in `sorted(...)` before iterating, though that would sacrifice the "lazy" memory benefit of the generator.

In [ ]:
# Get Llama 3.2 to answer

response = ollama.chat.completions.create(model=MODEL_LLAMA, messages=messages)
content = response.choices[0].message.content
display(Markdown(content))